In [1]:
import numpy as np
import pandas as pd
import os

# 1. 폴더 생성
output_dir = "./speaking"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def generate_rubric_scores(n_students, n_factors, target_mean, target_std, min_val, max_val):
    """
    논문의 총점 통계치를 바탕으로 하위 루브릭 항목별 점수를 생성하는 함수
    """
    # 학생별 목표 총점 생성 (정규분포 활용)
    total_scores = np.random.randn(n_students)
    total_scores = (total_scores - np.mean(total_scores)) / np.std(total_scores) * target_std + target_mean
    
    data = np.zeros((n_students, n_factors))
    for i in range(n_students):
        target_sum = total_scores[i]
        
        # 4개 요인에 무작위 분배 후 반올림 (각 5점 만점)
        row = np.random.rand(n_factors) + 0.5
        row = (row / np.sum(row)) * target_sum
        row = np.clip(np.round(row), min_val, max_val)
        
        # 총점 오차 조정
        diff = int(round(target_sum)) - int(np.sum(row))
        attempts = 0
        while diff != 0 and attempts < 500:
            idx = np.random.randint(n_factors)
            if diff > 0 and row[idx] < max_val:
                row[idx] += 1
                diff -= 1
            elif diff < 0 and row[idx] > min_val:
                row[idx] -= 1
                diff += 1
            attempts += 1
        data[i] = row
    return data.astype(int)

# 설정값: 총 36명, 4개 평가 요인 (각 5점 만점) [cite: 11, 147]
n_per_group = 18
n_factors = 4
factors = ['Fluency', 'Accuracy', 'Interaction', 'Content']

# ==========================================
# 2. 말하기 성취도 데이터 생성 (논문 표 10 통계치 반영) 
# ==========================================
# 실험집단 (Experimental)
spk_pre_exp = generate_rubric_scores(n_per_group, n_factors, 13.45, 1.82, 1, 5)
spk_post_exp = generate_rubric_scores(n_per_group, n_factors, 16.92, 1.65, 1, 5)

# 통제집단 (Control): 사전 동질성(p>.60) 및 사후 변화 없음 반영 
spk_pre_ctrl = generate_rubric_scores(n_per_group, n_factors, 13.40, 1.80, 1, 5)
spk_post_ctrl = generate_rubric_scores(n_per_group, n_factors, 13.55, 1.85, 1, 5)

# ==========================================
# 3. 데이터프레임 구성 및 저장
# ==========================================
def build_speaking_df(group_name, spk_pre, spk_post, start_id):
    df = pd.DataFrame()
    df['Student_ID'] = [f'{group_name[0].upper()}{str(i).zfill(2)}' for i in range(start_id, start_id + n_per_group)]
    df['Group'] = group_name
    
    for i, factor in enumerate(factors):
        df[f'Spk_Pre_{factor}'] = spk_pre[:, i]
        df[f'Spk_Post_{factor}'] = spk_post[:, i]
        
    # 총점 컬럼 추가
    df['Spk_Pre_Total'] = df[[f'Spk_Pre_{f}' for f in factors]].sum(axis=1)
    df['Spk_Post_Total'] = df[[f'Spk_Post_{f}' for f in factors]].sum(axis=1)
    return df

df_exp = build_speaking_df('Experimental', spk_pre_exp, spk_post_exp, 1)
df_ctrl = build_speaking_df('Control', spk_pre_ctrl, spk_post_ctrl, 1)

df_final = pd.concat([df_exp, df_ctrl], ignore_index=True)

# CSV 저장
file_path = os.path.join(output_dir, "simulated_speaking_data.csv")
df_final.to_csv(file_path, index=False)

print(f"✅ 말하기 데이터 생성 완료! 파일 경로: {file_path}")
print(f"📊 데이터 요약 (실험집단 사후 평균): {df_final[df_final['Group']=='Experimental']['Spk_Post_Total'].mean():.2f}")

✅ 말하기 데이터 생성 완료! 파일 경로: ./speaking/simulated_speaking_data.csv
📊 데이터 요약 (실험집단 사후 평균): 16.94
